# 02 — Feature Preparation

This notebook turns the QC'd dataset from `01_dataset_qc.ipynb` into a documented, final numeric/categorical feature configuration for modelling: element coverage, processing-feature availability, composition qualifier policy, and a categorical processing-route strategy.

It decides *which* features to use and *how* to resolve them; the actual resolution mechanics (`resolve_composition_column`, `build_preprocessor`, `apply_known_data_corrections`, `target_cohort_masks`) live in `modelling_lib.py`.

No model gets fit here. Every decision below comes from coverage/distribution evidence alone, never from how it would affect a fitted model's performance — otherwise "feature preparation" would end up outcome-driven rather than pre-specified.


## Imports

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "modelling" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from literature_pipeline.modelling import dataset_contract as dc
from literature_pipeline.modelling import modelling_lib as ml

# NOTE: this notebook is a thin experiment/analysis layer over
# literature_pipeline/modelling/dataset_contract.py and modelling_lib.py.
# No modelling/CV logic should be duplicated here.

## Load dataset independently (not carried over in-memory from 01)

Notebooks are independent — 01's corrections and cohort logic get re-applied here through the same shared `modelling_lib` functions, never assumed from a prior notebook's live session.


In [2]:
FINAL_DATASET_PATH = PROJECT_ROOT / "data" / "datasets_v2" / "production_run" / "cpp_dataset_v2.csv"

df_raw = pd.read_csv(FINAL_DATASET_PATH)
df = ml.apply_known_data_corrections(df_raw)
print(df.shape, "-- manual corrections applied:", df["manual_unit_correction_applied"].sum())

(256, 55) -- manual corrections applied: 1


## Target-specific composition coverage — observation-level and DOI-level

Computed on each target's `primary` cohort (`modelling_lib.target_cohort_masks`), not the whole raw dataset — a coverage figure over rows a target can't even use isn't the number that matters for feature selection. Both coverage measures are reported side by side, because an element present in many *rows* that all come from one *paper* is a very different situation from one spread across many papers, and row-coverage alone wouldn't tell the two apart.


In [3]:
# Shared function (literature_pipeline/modelling/modelling_lib.py) -- 03 calls the
# identical logic via ml.build_target_configs() below, never a notebook-local redefinition.
coverage_by_target = {}
for target in dc.TARGET_COLUMNS:
    masks = ml.target_cohort_masks(df, target)
    coverage_by_target[target] = ml.composition_coverage_table(df, masks["primary"])
    print(f"=== {target} (primary n={masks['primary'].sum()}, DOI={df.loc[masks['primary'],'doi'].nunique()}) ===")
    print(coverage_by_target[target].to_string(index=False))
    print()

=== UTS_MPa (primary n=155, DOI=36) ===
element  n_observations  observation_coverage  n_unique_DOI  DOI_coverage
     Si              94                 0.606            19         0.528
      C              87                 0.561            18         0.500
     Mn              84                 0.542            18         0.500
     Cr              67                 0.432            14         0.389
      P              58                 0.374            12         0.333
      S              58                 0.374            12         0.333
     Al              52                 0.335            12         0.333
     Ni              41                 0.265             8         0.222
      N              33                 0.213             5         0.139
     Mo              20                 0.129             6         0.167
     Nb              13                 0.084             3         0.083
      V              13                 0.084             2         0.05

## Zero-composition vs. partially-observed composition — two distinct missingness cases

A row with zero composition elements can't be modelled by any composition-using model at all — imputing a full alloy from nothing would fabricate the input rather than fill a gap. A row with at least one element but not all of them is a genuinely milder, more ordinary case of per-element missingness, and gets handled at CV time in 03 rather than resolved here.


In [4]:
zero_vs_partial = []
for target in dc.TARGET_COLUMNS:
    masks = ml.target_cohort_masks(df, target)
    primary_df = df[masks["primary"]]
    has_any = primary_df[dc.CORE_COMPOSITION_ELEMENTS].notna().any(axis=1)
    zero_vs_partial.append({
        "target": target,
        "primary_n": len(primary_df),
        "zero_composition_n": int((~has_any).sum()),
        "composition_present_n": int(has_any.sum()),
        "composition_present_DOI": primary_df.loc[has_any, "doi"].nunique(),
    })
pd.DataFrame(zero_vs_partial)

,target,primary_n,zero_composition_n,composition_present_n,composition_present_DOI
0,UTS_MPa,155,61,94,19
1,YS_MPa,153,58,95,20
2,elongation_pct,92,40,52,12


This decision is pre-specified, not threshold-tuned against any model result:

- Zero composition elements → excluded from every composition-using model. These rows stay usable for a composition-free baseline (target ~ processing/test-condition features only, if any survive the `processing_key` investigation below), and are never dropped from the dataset itself.
- At least one composition element → retained, with per-element missingness handled inside the CV pipeline in 03 (`modelling_lib.build_preprocessor`'s `SimpleImputer`), not resolved here.
- No arbitrary observation/DOI-coverage threshold decides which elements enter `NUMERIC_FEATURES`. The coverage tables above are the evidence, and the feature-set config below uses them directly — a `core` set of the best-covered elements, expandable later — rather than an invented single cutoff.


## Composition qualifier distribution — real values, not assumed categories


In [5]:
qualifier_summary = []
for element in dc.CORE_COMPOSITION_ELEMENTS:
    qcol = f"{element}_qualifier"
    vc = df[qcol].value_counts(dropna=False)
    # Does every non-"exact" qualifier correspond to a NULL value, or does any of them
    # carry an actual number that a naive "preserve all numeric values" policy would use?
    non_exact_with_value = df[(df[qcol].notna()) & (df[qcol] != "exact") & (df[element].notna())]
    qualifier_summary.append({
        "element": element,
        "qualifier_values": dict(vc[vc.index.notna()]) if vc.index.notna().any() else {},
        "non_exact_qualifier_WITH_a_numeric_value": len(non_exact_with_value),
    })
qual_df = pd.DataFrame(qualifier_summary)
qual_df

,element,qualifier_values,non_exact_qualifier_WITH_a_numeric_value
0,C,{'exact': 127},0
1,Mn,{'exact': 126},0
2,Si,{'exact': 135},0
3,Al,{'exact': 89},0
4,Cr,{'exact': 93},0
5,Ni,{'exact': 64},0
6,Mo,{'exact': 32},0
7,Nb,"{'exact': 33, 'not_reported': 9}",0
8,Ti,"{'exact': 22, 'not_reported': 10}",0
9,V,{'exact': 17},0


Qualifier handling was checked empirically, directly on the flat dataset's own `*_qualifier` columns (the structured value the dataset builder attached to each element, not re-derived from the audit trail). Every retained numerical composition value is labelled `"exact"`; the non-numerical qualifiers that do show up (`"not_reported"` for Mn/Si/Nb/Ti, `"balance"` for Al) never co-occur with an actual numeric value — `non_exact_qualifier_WITH_a_numeric_value == 0` for every element above. There's no `range`, `trace`, or `</>`-bound case anywhere in this dataset.

Since every present numeric value is already `"exact"`, `modelling_lib.resolve_composition_column`'s default strategy needs no customisation — no additional qualifier transformation was applied. That would only need revisiting if the corpus grows to include papers reporting ranges or traces.


## `processing_key` investigation — usable predictor, or DOI fingerprint?

Not assumed usable just because it's 255/297 non-null. A free-text or paper-specific identifier, one-hot-encoded as a "processing feature," would let a model learn which paper an observation came from rather than any real processing effect.


In [6]:
print("n unique processing_key values:", df["processing_key"].nunique(), "/ non-null rows:", df["processing_key"].notna().sum())
print()
key_doi_counts = df.dropna(subset=["processing_key"]).groupby("processing_key")["doi"].nunique().sort_values(ascending=False)
print("keys appearing in exactly 1 DOI (paper-specific):", int((key_doi_counts == 1).sum()), "/", len(key_doi_counts))
print("keys appearing in >1 DOI (potentially generalisable):", int((key_doi_counts > 1).sum()))
print()
print("top values by row count:")
print(df["processing_key"].value_counts().head(15).to_string())
print()
print("per-key DOI counts (top 15):")
print(key_doi_counts.head(15).to_string())

n unique processing_key values: 24 / non-null rows: 220

keys appearing in exactly 1 DOI (paper-specific): 20 / 24
keys appearing in >1 DOI (potentially generalisable): 4

top values by row count:
processing_key
__baseline__                                                                       57
intercritical annealing                                                            39
dq&p                                                                               18
intercritical annealing at 750°c and subsequent partitioning treatment at 400°c    17
tempering                                                                          14
b-q&p                                                                              11
ded-lb                                                                              8
quenching and partitioning (q&p)                                                    8
austempering                                                                        7
critical tempe

There are 26 unique `processing_key` values, and 22 of them (85%) appear in exactly one DOI — the large majority are paper-specific free-text descriptions (e.g. *"intercritical annealing at 750°C and subsequent partitioning treatment at 400°C"*, *"asymmetric cold rolling (60%) + annealing at 700°C"*), effectively unique identifiers rather than shared categories. Only 4 values genuinely recur across multiple papers: `__baseline__` (20 DOIs — an explicit "no special processing / as-received" sentinel, not a real processing description), `intercritical annealing` (8 DOIs), `quenching and partitioning (q&p)` (2 DOIs), and `dq&p` (2 DOIs).

Per the pre-specified rule — usable only if reliably reducible to standardised physical categories shared across multiple DOI — `processing_key` is not included as a raw one-hot categorical predictor in `FEATURE_SETS` below. One-hot-encoding 22 single-paper labels would let a model fit paper identity rather than a processing effect. `processing_identity_status` (confirmed above in 01) is explicitly a completeness/quality flag rather than a physical feature, and is excluded from `FEATURE_SETS` for the same reason.

This isn't a permanent verdict on processing as a modelling dimension — a hand-curated mapping of the 4 recurring values (plus `__baseline__` as its own explicit "no treatment" level) into a small `processing_category` feature would be a defensible follow-up. It's just not something to invent here without being asked for, given the scope of this notebook.


## Final feature-set configuration

Built once by the shared `modelling_lib.build_target_configs()`, not a notebook-local variable. 03 calls this exact same function on its own freshly-loaded dataframe — nothing is imported from this notebook's executed session, so the two stay genuinely independent.


In [7]:
TARGET_CONFIGS = ml.build_target_configs(df)  # default thresholds: min_observations=20, min_doi=5

for target, cfg in TARGET_CONFIGS.items():
    print(f"{target}: selection rule -- {cfg['selection_rule']}")
    print(f"  selected: {cfg['feature_sets']['composition']}")
    print()

UTS_MPa: selection rule -- n_observations >= 20 AND n_unique_DOI >= 5 (within this target's own primary cohort)
  selected: ['Si', 'C', 'Mn', 'Cr', 'P', 'S', 'Al', 'Ni', 'N', 'Mo']

YS_MPa: selection rule -- n_observations >= 20 AND n_unique_DOI >= 5 (within this target's own primary cohort)
  selected: ['C', 'Si', 'Mn', 'Cr', 'Al', 'P', 'S', 'Ni', 'N', 'Mo']

elongation_pct: selection rule -- n_observations >= 20 AND n_unique_DOI >= 5 (within this target's own primary cohort)
  selected: ['Si', 'C', 'Mn', 'P', 'S', 'Cr', 'Ni']



In [8]:
summary_rows = []
for target, cfg in TARGET_CONFIGS.items():
    cov = cfg["coverage_table"].set_index("element")
    for element in cfg["feature_sets"]["composition"]:
        summary_rows.append({
            "target": target,
            "primary_n": cfg["primary_n"],
            "primary_DOI": cfg["primary_DOI"],
            "selected_element": element,
            "element_n_observations": int(cov.loc[element, "n_observations"]),
            "element_n_DOI": int(cov.loc[element, "n_unique_DOI"]),
            "selection_rule": cfg["selection_rule"],
        })
final_summary_table = pd.DataFrame(summary_rows)
final_summary_table

,target,primary_n,primary_DOI,selected_element,element_n_observations,element_n_DOI,selection_rule
0,UTS_MPa,155,36,Si,94,19,n_observations >= 20 AND n_unique_DOI >= 5 (wi...
1,UTS_MPa,155,36,C,87,18,n_observations >= 20 AND n_unique_DOI >= 5 (wi...
2,UTS_MPa,155,36,Mn,84,18,n_observations >= 20 AND n_unique_DOI >= 5 (wi...
3,UTS_MPa,155,36,Cr,67,14,n_observations >= 20 AND n_unique_DOI >= 5 (wi...
4,UTS_MPa,155,36,P,58,12,n_observations >= 20 AND n_unique_DOI >= 5 (wi...
5,UTS_MPa,155,36,S,58,12,n_observations >= 20 AND n_unique_DOI >= 5 (wi...
6,UTS_MPa,155,36,Al,52,12,n_observations >= 20 AND n_unique_DOI >= 5 (wi...
7,UTS_MPa,155,36,Ni,41,8,n_observations >= 20 AND n_unique_DOI >= 5 (wi...
8,UTS_MPa,155,36,N,33,5,n_observations >= 20 AND n_unique_DOI >= 5 (wi...
9,UTS_MPa,155,36,Mo,20,6,n_observations >= 20 AND n_unique_DOI >= 5 (wi...


## Summary

Composition coverage was computed per target, on that target's `primary` cohort, at both observation- and DOI-level (`ml.composition_coverage_table` above) — no dataset-wide threshold was assumed in advance.

The selection rule, fixed before any model and never tuned against a result: an element is retained if it appears in at least 20 observations and at least 5 independent DOI within that target's own primary cohort (`modelling_lib.DEFAULT_MIN_OBSERVATIONS_FOR_FEATURE` / `DEFAULT_MIN_DOI_FOR_FEATURE`) — see `final_summary_table` above for the exact element/observation/DOI numbers behind every inclusion.

The library's default qualifier policy was confirmed correct empirically, directly on the flat dataset's own qualifier columns, so no custom strategy was needed.

`processing_key` was excluded as a predictor: 85% of its values are single-paper identifiers, a DOI-fingerprinting risk rather than a physical processing feature. As a consequence, the composition → composition+processing → composition+processing+test feature-set progression originally envisaged for RQ4 isn't currently supportable by standardised processing/test features. RQ4 is addressed instead in 04, by stratifying the same composition-only model's OOF errors by `processing_identity_status`/`test_conditions_status` (used to group results, never as model predictors) and by composition completeness.

Zero-composition rows are excluded from composition-using models rather than imputed from nothing; partial composition is retained and handled at CV time in 03.

`TARGET_CONFIGS = ml.build_target_configs(df)` is the reusable output — 03 calls the identical function, so no feature list or threshold gets redefined there.

No model was fit in this notebook, and no feature or threshold was chosen based on predictive performance.
